# Fatia 2 — Livro de Saída da Denver (1,59 milhão de linhas e janela `replaceWhere`)

Explora as camadas Delta gravadas por `jobs/fatia2_livro_saida.py` em `data/delta/`:
- `silver_livro_saida` (1.592.890 itens tipados em `DECIMAL(15,2)`, particionados por `ano, mes`);
- `gold_livro_saida_mensal` (30 competências reconciliadas ao centavo e comparadas com a perda do `float32`);
- Histórico (`DESCRIBE HISTORY`) comprovando que a janela de 3 meses (`replaceWhere`) trocou só 3 arquivos Parquet.

Dado real: este notebook não deve ser salvo com outputs contendo dados de clientes/NFs reais (constitution).

In [ ]:
from pyspark.sql import functions as F
from estudo.sessao import criar_sessao

spark = criar_sessao('notebook-livro-saida')
silver = spark.read.format('delta').load('/work/data/delta/silver_livro_saida')
gold = spark.read.format('delta').load('/work/data/delta/gold_livro_saida_mensal')
print('Itens na Silver:', silver.count())
silver.printSchema()

In [ ]:
# 1. Resumo por ano na Gold: DECIMAL(15,2) exato vs perda acumulada do float32
(
    gold.groupBy('ano')
    .agg(
        F.sum('itens').alias('itens'),
        F.sum('nfs_distintas').alias('nfs'),
        F.sum('vlr_total_nota_calc').cast('decimal(15,2)').alias('total_decimal_15_2'),
        F.sum('vlr_total_float32').cast('decimal(15,2)').alias('total_float32'),
        F.sum(F.abs(F.col('diferenca_float32'))).cast('decimal(15,2)').alias('erro_abs_float32_acumulado'),
    )
    .orderBy('ano')
    .show(truncate=False)
)

In [ ]:
# 2. As 10 competências com maior perda de centavos na soma em float32
(
    gold.select(
        'ano', 'mes', 'itens', 'nfs_distintas',
        'vlr_total_nota_calc', 'vlr_total_float32', 'diferenca_float32'
    )
    .orderBy(F.desc(F.abs(F.col('diferenca_float32'))))
    .show(10, truncate=False)
)

In [ ]:
# 3. Histórico do Delta na Silver: versão 0 (30 arquivos) vs versão 1 (replaceWhere de 3 meses: só 3 arquivos tocados)
(
    spark.sql("DESCRIBE HISTORY delta.`/work/data/delta/silver_livro_saida`")
    .selectExpr(
        'version',
        'timestamp',
        'operation',
        "operationParameters['predicate'] AS replaceWhere_predicate",
        "operationMetrics['numOutputRows'] AS linhas_gravadas",
        "operationMetrics['numFiles'] AS arquivos_gravados",
        "operationMetrics['numRemovedFiles'] AS arquivos_removidos"
    )
    .orderBy('version')
    .show(truncate=False)
)

In [ ]:
# 4. Time Travel: comparar a contagem e soma de 2026-09 na versão 0 (carga completa) e na versão 1 (após janela)
v0 = spark.read.format('delta').option('versionAsOf', 0).load('/work/data/delta/silver_livro_saida').where('ano = 2026 AND mes = 9')
v1 = spark.read.format('delta').option('versionAsOf', 1).load('/work/data/delta/silver_livro_saida').where('ano = 2026 AND mes = 9')
print('v0 (2026-09):', v0.agg(F.count('*'), F.sum('vlr_total_nota_calc')).first())
print('v1 (2026-09):', v1.agg(F.count('*'), F.sum('vlr_total_nota_calc')).first())